In [ ]:
pip install transformers tensorflow sentencepiece torch accelerate -U

In [ ]:
import pandas as pd
import numpy as np
from tqdm import tqdm
import pickle
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score
import torch
from transformers import TrainingArguments, Trainer
from transformers import BertTokenizer, BertForSequenceClassification
from transformers import EarlyStoppingCallback
import copy
from numpy import unique
from sklearn.preprocessing import LabelEncoder
import math
from transformers import XLMRobertaForSequenceClassification, XLMRobertaTokenizer
from sklearn.metrics import mean_squared_error
import matplotlib.pyplot as plt
import os
import torch
from huggingface_hub import from_pretrained_keras
from collections import Counter
from sklearn.metrics import mean_absolute_error

In [ ]:
#pandas settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
def map_target_var(award_price_divisions, y):
  y_encoded = []

  for i in range(len(y)):
    for j, interval in enumerate(award_price_divisions.keys()):
      lower_bound = award_price_divisions[interval][0]
      upper_bound = award_price_divisions[interval][1]

      if y[i] >= lower_bound and y[i] < upper_bound:
        y_encoded.append(j)
      else:
        continue

  #check for remaining values at the higher end of the array
  left_over_cases = len(y) - len(y_encoded)
  if left_over_cases > 0:
    for i in range(0, left_over_cases):
      y_encoded.append(j)

  y_encoded = np.array(y_encoded)

  return y_encoded

In [ ]:
#we need to get a better feeling for the distribution accros award price "classes", is there a distribution to be found which could improve accuracy, propagating to the final model in MAE
def award_price_distribution(y, amount_of_classes: int, make_labels):
  if make_labels == True:
    award_price_divisions = {}
    y_sorted = np.sort(y)
    length_y = len(y_sorted)
    length_class = int(length_y / amount_of_classes)
    lower_index = 0
    for i in range(1, amount_of_classes):
      upper_index = length_class * i
      lower_value = y_sorted[lower_index]
      upper_value = y_sorted[upper_index]
      print("lower index: {}, lower value {}, upper index {}, upper value {}".format(lower_index, lower_value, upper_index, upper_value))
      award_price_divisions[str(lower_value) + "-" + str(upper_value)] = [lower_value, upper_value]
      print("length of class = {}".format(len(y_sorted[lower_index:upper_index])))
      lower_index += 1 * length_class

    y = map_target_var(award_price_divisions, y)
    return y
  else:
    print("no encoding or labels were made for the target variable")
    return y

In [ ]:
def plot_metrics(ledger, save_path = None):
  #model results is a dict from a keras object
  model_results = ledger.history
  plt.figure(figsize=(10, 10))

  # Create the top row of subplots
  ax1 = plt.subplot(3, 1, 1)
  ax2 = plt.subplot(3, 1, 2)
  ax3 = plt.subplot(3, 1, 3)
  axes = [ax1, ax2, ax3]
  for i, key in enumerate(list(model_results.keys())[:int(0.5*len(model_results.keys()))]):
    loss_train = model_results[key]
    loss_val   = model_results["val_" + key]
    epochs = range(0,len(loss_train))

    axes[i].plot(epochs, loss_train, "g", label = "Training {}".format(key))
    axes[i].plot(epochs, loss_val, "b", label = "Validation {}".format("val_" + key))
    axes[i].title.set_text("Training and validation {}".format(key))
    axes[i].set_xlabel("Epochs")
    axes[i].set_ylabel("{}".format(key))
    axes[i].legend()
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, format='png')

In [ ]:
#let's filter on the outliers and see how model performance increases
def filter_outliers(df, upper, lower, target_col = "AWARD_VALUE_EURO_FIN_1"):
    """This function only works on numerical columns"""
    data_array = np.sort(df[target_col].to_numpy())
    cut_off_low_indice = math.floor(lower * len(data_array))
    cut_off_high_indice = math.floor(upper * len(data_array)) - 1
    low_amount = data_array[cut_off_low_indice]
    high_amount = data_array[cut_off_high_indice]

    outlier_indices = []

    for i in range(0, len(df)):
        if df[target_col].iloc[i] > high_amount:
            outlier_indices.append(i)
        elif df[target_col].iloc[i] < low_amount:
            outlier_indices.append(i)
        else:
            continue

    print("{} rows were dropped because of outliers".format(len(outlier_indices)))
    df = df.drop(labels = outlier_indices)
    return df

In [ ]:
#load the data
df_original = pd.read_pickle("/content/drive/MyDrive/Thesis/df_merged_dataset_clean").reset_index()
df_original["short description"] = df_original["short description"].astype(str)

In [ ]:
df = copy.deepcopy(df_original[:15000])

In [ ]:
df = filter_outliers(df, upper=0.90, lower = 0.20)

In [ ]:
df.head()

In [ ]:
df["short description"][:5].tolist()

In [ ]:
def train_validate_test_split(df, train, validate, goal):
    target_col = "AWARD_VALUE_EURO_FIN_1"
    text_col = "short description"
    train_indice = int(train * len(df))
    validate_indice = train_indice + int(validate * len(df))

    train_set = df[:train_indice]
    val_set = df[train_indice:validate_indice]
    test_set = df[validate_indice:]

    if goal == "all":
        X_train = train_set.drop(columns=[target_col]).values
        y_train = train_set[target_col].values

        X_val = val_set.drop(columns=[target_col]).values
        y_val = val_set[target_col].values

        X_test = test_set.drop(columns=[target_col]).values
        y_test = test_set[target_col].values

    elif goal == "text":
        X_train = train_set[text_col].tolist()
        y_train = train_set[target_col].values

        X_val = val_set[text_col].tolist()
        y_val = val_set[target_col].values

        X_test = test_set[text_col].tolist()
        y_test = test_set[target_col].values

    return X_train, y_train, X_val, y_val, X_test, y_test

In [ ]:
#CREATE TRAIN, VAL TEST SPLIT FOR THE TEXT. no two lots from the same tender can be together
X_train, y_train, X_val, y_val, X_test, y_test = train_validate_test_split(df, 0.6, 0.2, "text")

In [ ]:
print(len(X_train), len(X_val), len(X_test))

In [ ]:
# Specify the model name or path
model_name = "xlm-roberta-base"
# Load the model and tokenizer
model = XLMRobertaForSequenceClassification.from_pretrained(model_name, num_labels=1)
tokenizer = XLMRobertaTokenizer.from_pretrained(model_name)

In [ ]:
X_train_tokenized = tokenizer(X_train, padding=True, truncation=True, max_length=512)
X_val_tokenized = tokenizer(X_val, padding=True, truncation=True, max_length=512)
X_test_tokenized = tokenizer(X_test, padding=True, truncation=True, max_length=512)

In [ ]:
class Dataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels=None):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        if self.labels is not None and len(self.labels) > 0:
            item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.encodings["input_ids"])

def compute_metrics(p):
    predictions, labels = p.predictions, p.label_ids
    mse = mean_squared_error(labels, predictions)
    mae = mean_absolute_error(labels, predictions)

    return {"mse": mse, "mae": mae}

y_train = torch.tensor(y_train, dtype=torch.float)
y_val = torch.tensor(y_val, dtype=torch.float)
y_test = torch.tensor(y_test, dtype=torch.float)

train_dataset = Dataset(X_train_tokenized, y_train)
val_dataset = Dataset(X_val_tokenized, y_val)
test_dataset = Dataset(X_test_tokenized, y_test)

In [ ]:
#! pip install -U accelerate
#! pip install -U transformers

In [ ]:
import torch
from transformers import Trainer, TrainingArguments

# Define Trainer
args = TrainingArguments(
    output_dir="output",
    evaluation_strategy="steps",
    eval_steps=500,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=5,
    seed=0,
    load_best_model_at_end=True,
    learning_rate=0.0001
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

# Train pre-trained model
training_results = trainer.train()
# Save the model
trainer.save_model("your_saved_model_directory")

In [ ]:
training_results

In [ ]:
#with open("/content/drive/MyDrive/Thesis/results_transformer_model_only.pickle", "wb") as f:
#  pickle.dump(training_results, f)

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

class PyTorchModel(nn.Module):
    def __init__(self, input_size, hidden_size1, hidden_size2):
        super(PyTorchModel, self).__init__()
        self.fc1 = nn.Linear(input_size, hidden_size1)
        self.relu1 = nn.ReLU()
        self.dropout1 = nn.Dropout(p=0.2)
        self.fc2 = nn.Linear(hidden_size1, hidden_size2)
        self.relu2 = nn.ReLU()
        self.dropout2 = nn.Dropout(p=0.2)
        self.fc3 = nn.Linear(hidden_size2, hidden_size1)
        self.regression_layer = nn.Linear(hidden_size1, 1)

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu1(x)
        x = self.dropout1(x)
        x = self.fc2(x)
        x = self.relu2(x)
        x = self.dropout2(x)
        x = self.fc3(x)
        regression_output = self.regression_layer(x)
        return regression_output

# Instantiate your PyTorch model
input_size = 10  # Update with your actual input dimension
hidden_size1 = 32
hidden_size2 = 8

pytorch_model = PyTorchModel(input_size, hidden_size1, hidden_size2)

# You can print the model summary if needed
print(pytorch_model)
